In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from arch import arch_model
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ==========================
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ (BTC-USD)
# ==========================
print("Скачиваем данные BTC-USD за последние 5 лет...")
ticker = "BTC-USD"
data = yf.download(ticker, start="2019-01-01", end="2024-12-31", progress=False)['Adj Close']

# TODO 1.1: Рассчитайте ежедневные логарифмические доходности.
# Используйте np.log и метод .shift(1). Удалите первое наблюдение (NaN).
returns = # <ВАШ_КОД>

# TODO 1.2: Разделите временной ряд на обучающую (80%), валидационную (10%) и тестовую (10%) выборки.
# Сохраните индексы дат для каждой выборки для последующей визуализации.
split_train = int(0.8 * len(returns))
split_val = int(0.9 * len(returns))
returns_train = # <ВАШ_КОД>
returns_val = # <ВАШ_КОД>
returns_test = # <ВАШ_КОД>

# Визуализируем данные
plt.figure(figsize=(12,5))
plt.plot(returns_train.index, returns_train.values, label='Train', alpha=0.7)
plt.plot(returns_val.index, returns_val.values, label='Validation', alpha=0.7)
plt.plot(returns_test.index, returns_test.values, label='Test', alpha=0.7)
plt.title('Логарифмические доходности BTC-USD')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ==========================
# 2. КЛАССИЧЕСКАЯ МОДЕЛЬ GARCH(1,1)
# ==========================
# Обучаем GARCH(1,1) на обучающей выборке
print("\nОбучаем GARCH(1,1)...")
garch_model = arch_model(returns_train * 100, vol='Garch', p=1, q=1, dist='normal')
garch_fitted = garch_model.fit(update_freq=5, disp='off')

# Сохраняем прогнозные волатильности (условные дисперсии)
garch_forecast = garch_fitted.forecast(horizon=1, reindex=False)
# TODO 2.1: Извлеките прогнозируемую волатильность (стандартное отклонение) для каждого дня валидационной и тестовой выборок.
# Подсказка: garch_forecast.variance содержит дисперсии, но нужно правильно индексировать.
garch_vol_val = # <ВАШ_КОД (для валидации)>
garch_vol_test = # <ВАШ_КОД (для теста)>

# TODO 2.2: Для тестовой выборки рассчитайте однодневный VaR на уровне доверия 99% и 95%.
# Формула VaR = -нормальный_квантиль * волатильность (для длинной позиции).
# Используйте scipy.stats.norm.ppf.
from scipy.stats import norm
VaR_99_garch = # <ВАШ_КОД>
VaR_95_garch = # <ВАШ_КОД>

# ==========================
# 3. ПОДГОТОВКА ДАННЫХ ДЛЯ LSTM/GRU
# ==========================
# Для прогнозирования волатильности будем использовать скользящее окно доходностей.
# Целевая переменная: волатильность (например, стандартное отклонение доходностей за следующие 5 дней) или непосредственно следующая доходность.
# В данном случае спрогнозируем абсолютное значение будущей доходности как proxy волатильности.

# TODO 3.1: Создайте последовательности (X, y) для обучения нейросети.
# Пусть X — это окно из последних N дней исторических доходностей (например, N=60).
# y — абсолютное значение доходности на следующий день (|r_{t+1}|).
# Используйте numpy array для формирования X и y.
window_size = 60
def create_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(np.abs(data[i]))   # предсказываем абсолютную доходность (proxy волатильности)
    return np.array(X), np.array(y)

# Применяем к доходностям тренировочной выборки
X_train, y_train = # <ВАШ_КОД>
X_val, y_val = # <ВАШ_КОД (на returns_val)>
X_test, y_test = # <ВАШ_КОД (на returns_test)>

# TODO 3.2: Нормализуйте входные данные (X) с помощью StandardScaler.
# Обучите scaler только на тренировочных данных, затем преобразуйте валидационные и тестовые.
scaler = StandardScaler()
X_train_scaled = # <ВАШ_КОД>
X_val_scaled = # <ВАШ_КОД>
X_test_scaled = # <ВАШ_КОД>

# Для LSTM вход должен быть 3D: (samples, timesteps, features). У нас features=1.
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val_lstm = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

# ==========================
# 4. ПОСТРОЕНИЕ И ОБУЧЕНИЕ LSTM
# ==========================
# TODO 4.1: Создайте модель LSTM с двумя слоями LSTM (по 50 нейронов), Dropout(0.2), затем Dense(1).
# Используйте активацию 'relu' для LSTM, 'linear' для выходного слоя.
# Компилируйте с оптимизатором 'adam' и функцией потерь 'mse'.
lstm_model = # <ВАШ_КОД>

# TODO 4.2: Обучите модель с использованием EarlyStopping (patience=10) на валидационных данных.
# Установите количество эпох (например, 100) и batch_size=32.
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_lstm = # <ВАШ_КОД: lstm_model.fit(...)>

# TODO 4.3: Сделайте прогноз абсолютной доходности на тестовой выборке.
pred_abs_return_lstm = # <ВАШ_КОД: model.predict(X_test_lstm)>

# ==========================
# 5. ПОСТРОЕНИЕ И ОБУЧЕНИЕ GRU (для сравнения)
# ==========================
# TODO 5.1: Создайте аналогичную модель, но с использованием слоев GRU (два слоя по 50 нейронов, Dropout(0.2), Dense(1)).
gru_model = # <ВАШ_КОД>

# TODO 5.2: Скомпилируйте и обучите GRU с теми же параметрами.
history_gru = # <ВАШ_КОД: gru_model.fit(...)>

# TODO 5.3: Получите прогнозы на тесте.
pred_abs_return_gru = # <ВАШ_КОД>

# ==========================
# 6. РАСЧЕТ VaR ДЛЯ НЕЙРОСЕТЕВЫХ МОДЕЛЕЙ
# ==========================
# Нейросеть предсказывает абсолютную доходность. Используем её как прогноз волатильности (при условии нулевого среднего).
# TODO 6.1: Для тестовой выборки рассчитайте VaR_99 и VaR_95 для LSTM и GRU.
# Формула: VaR = - norm.ppf(confidence) * predicted_volatility (где predicted_volatility = pred_abs_return).
VaR_99_lstm = # <ВАШ_КОД>
VaR_95_lstm = # <ВАШ_КОД>
VaR_99_gru = # <ВАШ_КОД>
VaR_95_gru = # <ВАШ_КОД>

# ==========================
# 7. БЭКТЕСТИНГ VaR (КОЛИЧЕСТВО ПРОБИЙ)
# ==========================
# Для каждой модели посчитаем количество дней, когда фактический убыток (отрицательная доходность) превышал VaR.
# Фактический убыток для длинной позиции = -return_t (если return_t отрицательный, то убыток положительный).
actual_loss = -returns_test.values[window_size:]  # обрезаем до длины прогнозов

# TODO 7.1: Реализуйте функцию нарушения VaR (exceedance).
# Вход: массив VaR (положительные числа), массив фактических убытков. Выход: доля нарушений.
def violation_ratio(var_values, actual_losses):
    # <ВАШ_КОД>

# Для GARCH (убедитесь, что длины совпадают)
actual_loss_garch = actual_loss  # так как GARCH дал прогнозы на тест той же длины

print("=== Доля нарушений VaR (тестовая выборка) ===")
print(f"GARCH(1,1) 99%: {violation_ratio(VaR_99_garch, actual_loss_garch):.4f} (ожидалось 0.01)")
print(f"GARCH(1,1) 95%: {violation_ratio(VaR_95_garch, actual_loss_garch):.4f} (ожидалось 0.05)")
print(f"LSTM 99%: {violation_ratio(VaR_99_lstm.flatten(), actual_loss):.4f}")
print(f"LSTM 95%: {violation_ratio(VaR_95_lstm.flatten(), actual_loss):.4f}")
print(f"GRU 99%: {violation_ratio(VaR_99_gru.flatten(), actual_loss):.4f}")
print(f"GRU 95%: {violation_ratio(VaR_95_gru.flatten(), actual_loss):.4f}")

# ==========================
# 8. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ==========================
# График 1: Сравнение предсказанной волатильности (абс. доходности) на тесте
plt.figure(figsize=(14,5))
plt.plot(returns_test.index[window_size:], y_test, label='Actual |return|', alpha=0.6)
plt.plot(returns_test.index[window_size:], pred_abs_return_lstm, label='LSTM predicted', alpha=0.8)
plt.plot(returns_test.index[window_size:], pred_abs_return_gru, label='GRU predicted', alpha=0.8)
plt.title('Прогноз абсолютной доходности (proxy волатильности)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# График 2: Сравнение VaR 99% на тесте (выбрать одну модель для наглядности)
plt.figure(figsize=(14,5))
plt.plot(returns_test.index[window_size:], returns_test.values[window_size:], label='Доходность', alpha=0.6)
plt.plot(returns_test.index[window_size:], -VaR_99_garch, label='GARCH VaR 99%', linestyle='--', color='red')
plt.plot(returns_test.index[window_size:], -VaR_99_lstm.flatten(), label='LSTM VaR 99%', linestyle='--', color='blue')
plt.fill_between(returns_test.index[window_size:], -VaR_99_garch, -VaR_99_lstm.flatten(), alpha=0.2)
plt.title('Сравнение VaR 99%: GARCH vs LSTM')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# График 3: Кумулятивные нарушения VaR (кумулятивная сумма превышений)
# TODO 8.1: Постройте ступенчатый график кумулятивного количества нарушений для каждой модели.
# Это поможет увидеть, не сгруппированы ли нарушения (что плохо для модели).